### Tumor Segmentation Notebook

#### Load Librairies

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from unet import UNet


#### Defining Training Hyperparameters 

In [2]:
# Data and training configuration
DATA_ROOT = Path('/Users/felixdelisle/Dev/models/datasets/brisc2025/segmentation_task')
TRAIN_IMAGES = DATA_ROOT / 'train' / 'images'
TRAIN_MASKS = DATA_ROOT / 'train' / 'masks'
VAL_IMAGES = DATA_ROOT / 'test' / 'images'
VAL_MASKS = DATA_ROOT / 'test' / 'masks'

BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS = 100
NUM_WORKERS = 0
IMAGE_SIZE = (256, 256)
NUM_CLASSES = 1
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


#### Dataset Preparation

In [3]:
def tversky_loss(pred, target, alpha=0.3, beta=0.7, gamma=2 ,eps=1e-6):
    pred_prob = torch.sigmoid(pred)
    pred_flat = pred_prob.reshape(pred.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1).float()

    tp = (pred_flat * target_flat).sum(dim=1)
    fp = ((1.0 - target_flat) * pred_flat).sum(dim=1)
    fn = (target_flat * (1.0 - pred_flat)).sum(dim=1)

    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return (1.0 - tversky.mean())**(1/gamma)


def bce_dice_tversky_loss(pred, target, bce_weight=2.0, tversky_weight=1.0):
    bce = nn.BCEWithLogitsLoss()
    return bce_weight * bce(pred, target) + tversky_weight * tversky_loss(pred, target)


def compute_iou_per_class(pred_logits, target, threshold=0.5):
    probs = torch.sigmoid(pred_logits)
    pred_mask = (probs >= threshold).float()
    target = target.float()
    metrics = {}

    for cls in [0, 1]:
        pred_cls = (pred_mask == cls).float()
        target_cls = (target == cls).float()
        intersection = (pred_cls * target_cls).sum().item()
        union = pred_cls.sum().item() + target_cls.sum().item() - intersection
        metrics[f'class_{cls}'] = (intersection / (union + 1e-6)) if union > 0 else 1.0

    return metrics


model = UNet(in_channels=3, out_channels=1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(model)


UNet(
  (encoder): Encoder(
    (encoder1): EncoderBlock(
      (conv1): ConvBlock(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (activation): SiLU(inplace=True)
      )
      (conv2): ConvBlock(
        (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (activation): SiLU(inplace=True)
      )
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (encoder2): EncoderBlock(
      (conv1): ConvBlock(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (activation): SiLU(inplace=True)
      )
      (conv2): ConvBlock(
      

In [4]:
class BRISCSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.target_transform = target_transform

        self.image_files = sorted(
            p for p in self.image_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
        )
        self.mask_files = {p.stem: p for p in self.mask_dir.iterdir() if p.is_file() and p.suffix.lower() == '.png'}

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = self.image_files[idx]
        mask_path = self.mask_files.get(image_path.stem)
        if mask_path is None:
            raise FileNotFoundError(f'Mask not found for image {image_path.name}')

        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
        else:
            mask = np.array(mask, dtype=np.float32) / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)
            if image.shape[-2:] != mask.shape[-2:]:
                mask = torch.nn.functional.interpolate(mask.unsqueeze(0), size=image.shape[-2:], mode='nearest').squeeze(0)

        if image.shape[0] != 3:
            image = image[:3, :, :]

        return image, mask


def pad_to_size(image, target_size=(256, 256)):
    image = np.array(image)
    if image.ndim == 2:
        image = image[..., None]
    h, w = image.shape[:2]
    target_h, target_w = target_size
    pad_h = max(0, target_h - h)
    pad_w = max(0, target_w - w)
    if pad_h or pad_w:
        padded = np.zeros((target_h, target_w, image.shape[2]), dtype=image.dtype)
        padded[:h, :w, ...] = image
        return padded
    return image


def to_tensor(x):
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x)
    if isinstance(x, Image.Image):
        x = np.array(x)
        x = torch.from_numpy(x).permute(2, 0, 1)
    return x.float() / 255.0 if x.dtype == torch.float32 or x.dtype == torch.uint8 else x


def train_transform(image):
    image = np.array(image.resize(IMAGE_SIZE, resample=Image.BILINEAR))
    image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    return image


def mask_transform(mask):
    mask = np.array(mask.resize(IMAGE_SIZE, resample=Image.NEAREST), dtype=np.float32) / 255.0
    mask = torch.from_numpy(mask).unsqueeze(0)
    return mask


train_ds = BRISCSegmentationDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS,
    transform=train_transform,
    target_transform=mask_transform,
)
val_ds = BRISCSegmentationDataset(
    VAL_IMAGES,
    VAL_MASKS,
    transform=train_transform,
    target_transform=mask_transform,
)

print(f'Train samples: {len(train_ds)}')
print(f'Validation samples: {len(val_ds)}')

sample_image, sample_mask = train_ds[0]
print('Image shape:', tuple(sample_image.shape))
print('Mask shape:', tuple(sample_mask.shape))
print('Mask min/max:', sample_mask.min().item(), sample_mask.max().item())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


Train samples: 3933
Validation samples: 860
Image shape: (3, 256, 256)
Mask shape: (1, 256, 256)
Mask min/max: 0.0 1.0


#### Training loop

In [5]:
cwd = Path.cwd()

if cwd.name == "UNet":
    CHECKPOINT_DIR = cwd / "checkpoints"
elif (cwd / "UNet").is_dir():
    CHECKPOINT_DIR = cwd / "UNet" / "checkpoints"
else:
    raise RuntimeError(f"Impossible de trouver le dossier UNet depuis {cwd}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def train_one_epoch(model, loader, optimizer, device, log_every=10):
    model.train()
    running_loss = 0.0
    running_iou = {'class_0': 0.0, 'class_1': 0.0}
    total_seen = 0

    for batch_idx, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, dtype=torch.float32)
        targets = targets.to(device, dtype=torch.float32)

        optimizer.zero_grad()
        logits = model(images)
        loss = bce_dice_tversky_loss(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_seen += batch_size

        batch_iou = compute_iou_per_class(logits, targets)
        for key in running_iou:
            running_iou[key] += batch_iou[key] * batch_size

        if batch_idx % log_every == 0 or batch_idx == len(loader):
            avg_loss = running_loss / total_seen
            avg_iou = {key: running_iou[key] / total_seen for key in running_iou}
            print(
                f"  Batch {batch_idx}/{len(loader)} - loss={avg_loss:.4f} "
                f"- IoU[class_0]={avg_iou['class_0']:.4f} "
                f"- IoU[class_1]={avg_iou['class_1']:.4f}"
            )

    return running_loss / total_seen, {key: running_iou[key] / total_seen for key in running_iou}


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    running_iou = {'class_0': 0.0, 'class_1': 0.0}
    total_seen = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, dtype=torch.float32)
            targets = targets.to(device, dtype=torch.float32)

            logits = model(images)
            loss = bce_dice_tversky_loss(logits, targets)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_seen += batch_size

            batch_iou = compute_iou_per_class(logits, targets)
            for key in running_iou:
                running_iou[key] += batch_iou[key] * batch_size

    return running_loss / total_seen, {key: running_iou[key] / total_seen for key in running_iou}

best_val_loss = float('inf')

history = {
    "train_loss": [],
    "val_loss": [],
    "train_iou_class_0": [],
    "train_iou_class_1": [],
    "val_iou_class_0": [],
    "val_iou_class_1": [],
}

for epoch in tqdm(range(EPOCHS)):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, DEVICE, log_every=10)
    val_loss, val_iou = evaluate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou_class_0"].append(train_iou['class_0'])
    history["train_iou_class_1"].append(train_iou['class_1'])
    history["val_iou_class_0"].append(val_iou['class_0'])
    history["val_iou_class_1"].append(val_iou['class_1'])

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"train_loss={train_loss:.4f} - val_loss={val_loss:.4f} - "
        f"train_iou[class_0]={train_iou['class_0']:.4f} - train_iou[class_1]={train_iou['class_1']:.4f} - "
        f"val_iou[class_0]={val_iou['class_0']:.4f} - val_iou[class_1]={val_iou['class_1']:.4f}"
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        }, CHECKPOINT_DIR / "best_model.pt")
    else:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        }, CHECKPOINT_DIR / "current_model.pt")


print("Training finished.")
print(history)


  0%|          | 0/100 [00:00<?, ?it/s]

  Batch 10/246 - loss=2.4484 - IoU[class_0]=0.3185 - IoU[class_1]=0.0103
  Batch 20/246 - loss=2.4080 - IoU[class_0]=0.3693 - IoU[class_1]=0.0103
  Batch 30/246 - loss=2.3786 - IoU[class_0]=0.4444 - IoU[class_1]=0.0113
  Batch 40/246 - loss=2.3498 - IoU[class_0]=0.5368 - IoU[class_1]=0.0163
  Batch 50/246 - loss=2.3243 - IoU[class_0]=0.6118 - IoU[class_1]=0.0243
  Batch 60/246 - loss=2.3019 - IoU[class_0]=0.6662 - IoU[class_1]=0.0353
  Batch 70/246 - loss=2.2810 - IoU[class_0]=0.7084 - IoU[class_1]=0.0469
  Batch 80/246 - loss=2.2648 - IoU[class_0]=0.7398 - IoU[class_1]=0.0580
  Batch 90/246 - loss=2.2522 - IoU[class_0]=0.7633 - IoU[class_1]=0.0654
  Batch 100/246 - loss=2.2419 - IoU[class_0]=0.7832 - IoU[class_1]=0.0700
  Batch 110/246 - loss=2.2325 - IoU[class_0]=0.7993 - IoU[class_1]=0.0758
  Batch 120/246 - loss=2.2233 - IoU[class_0]=0.8132 - IoU[class_1]=0.0839
  Batch 130/246 - loss=2.2147 - IoU[class_0]=0.8251 - IoU[class_1]=0.0926
  Batch 140/246 - loss=2.2069 - IoU[class_0]=0.

  1%|          | 1/100 [03:30<5:47:06, 210.37s/it]

Epoch 1/100 - train_loss=2.1513 - val_loss=2.1361 - train_iou[class_0]=0.8929 - train_iou[class_1]=0.1340 - val_iou[class_0]=0.9598 - val_iou[class_1]=0.1762
  Batch 10/246 - loss=2.0412 - IoU[class_0]=0.9715 - IoU[class_1]=0.2176
  Batch 20/246 - loss=2.0339 - IoU[class_0]=0.9711 - IoU[class_1]=0.2139
  Batch 30/246 - loss=2.0297 - IoU[class_0]=0.9731 - IoU[class_1]=0.2190
  Batch 40/246 - loss=2.0287 - IoU[class_0]=0.9718 - IoU[class_1]=0.2184
  Batch 50/246 - loss=2.0260 - IoU[class_0]=0.9720 - IoU[class_1]=0.2267
  Batch 60/246 - loss=2.0241 - IoU[class_0]=0.9722 - IoU[class_1]=0.2301
  Batch 70/246 - loss=2.0208 - IoU[class_0]=0.9730 - IoU[class_1]=0.2330
  Batch 80/246 - loss=2.0164 - IoU[class_0]=0.9738 - IoU[class_1]=0.2384
  Batch 90/246 - loss=2.0113 - IoU[class_0]=0.9746 - IoU[class_1]=0.2478
  Batch 100/246 - loss=2.0077 - IoU[class_0]=0.9751 - IoU[class_1]=0.2506
  Batch 110/246 - loss=2.0035 - IoU[class_0]=0.9757 - IoU[class_1]=0.2576
  Batch 120/246 - loss=2.0003 - IoU[c

  2%|▏         | 2/100 [07:02<5:45:30, 211.53s/it]

...

Epoch 98/100 - train_loss=0.1634 - val_loss=0.5109 - train_iou[class_0]=0.9977 - train_iou[class_1]=0.8574 - val_iou[class_0]=0.9933 - val_iou[class_1]=0.6780
  Batch 10/246 - loss=0.1545 - IoU[class_0]=0.9978 - IoU[class_1]=0.8633
  Batch 20/246 - loss=0.1632 - IoU[class_0]=0.9978 - IoU[class_1]=0.8640
  Batch 30/246 - loss=0.1635 - IoU[class_0]=0.9977 - IoU[class_1]=0.8609
  Batch 40/246 - loss=0.1617 - IoU[class_0]=0.9977 - IoU[class_1]=0.8602
  Batch 50/246 - loss=0.1608 - IoU[class_0]=0.9977 - IoU[class_1]=0.8595
  Batch 60/246 - loss=0.1598 - IoU[class_0]=0.9977 - IoU[class_1]=0.8606
  Batch 70/246 - loss=0.1597 - IoU[class_0]=0.9977 - IoU[class_1]=0.8610
  Batch 80/246 - loss=0.1587 - IoU[class_0]=0.9977 - IoU[class_1]=0.8614
  Batch 90/246 - loss=0.1595 - IoU[class_0]=0.9977 - IoU[class_1]=0.8600
  Batch 100/246 - loss=0.1616 - IoU[class_0]=0.9976 - IoU[class_1]=0.8594
  Batch 110/246 - loss=0.1623 - IoU[class_0]=0.9976 - IoU[class_1]=0.8591
  Batch 120/246 - loss=0.1622 - IoU[

 99%|█████████▉| 99/100 [5:54:10<03:32, 212.81s/it]

Epoch 99/100 - train_loss=0.1641 - val_loss=0.5159 - train_iou[class_0]=0.9977 - train_iou[class_1]=0.8568 - val_iou[class_0]=0.9932 - val_iou[class_1]=0.6742
  Batch 10/246 - loss=0.1569 - IoU[class_0]=0.9977 - IoU[class_1]=0.8686
  Batch 20/246 - loss=0.1590 - IoU[class_0]=0.9977 - IoU[class_1]=0.8651
  Batch 30/246 - loss=0.1582 - IoU[class_0]=0.9977 - IoU[class_1]=0.8634
  Batch 40/246 - loss=0.1571 - IoU[class_0]=0.9977 - IoU[class_1]=0.8649
  Batch 50/246 - loss=0.1570 - IoU[class_0]=0.9977 - IoU[class_1]=0.8634
  Batch 60/246 - loss=0.1587 - IoU[class_0]=0.9977 - IoU[class_1]=0.8635
  Batch 70/246 - loss=0.1587 - IoU[class_0]=0.9977 - IoU[class_1]=0.8623
  Batch 80/246 - loss=0.1603 - IoU[class_0]=0.9977 - IoU[class_1]=0.8619
  Batch 90/246 - loss=0.1602 - IoU[class_0]=0.9977 - IoU[class_1]=0.8616
  Batch 100/246 - loss=0.1604 - IoU[class_0]=0.9977 - IoU[class_1]=0.8604
  Batch 110/246 - loss=0.1604 - IoU[class_0]=0.9977 - IoU[class_1]=0.8604
  Batch 120/246 - loss=0.1607 - IoU[

100%|██████████| 100/100 [5:57:43<00:00, 214.63s/it]

Epoch 100/100 - train_loss=0.1624 - val_loss=0.4942 - train_iou[class_0]=0.9977 - train_iou[class_1]=0.8577 - val_iou[class_0]=0.9933 - val_iou[class_1]=0.6810
Training finished.
{'train_loss': [2.151324070654781, 1.9529166861766172, 1.7730738858124273, 1.623471292441897, 1.4809861647515936, 1.3535050198591123, 1.2475473683752725, 1.1522734173072253, 1.0671804719016111, 0.9894534033459621, 0.9168140002555469, 0.8433341829794344, 0.7749882086281736, 0.7089404314491181, 0.6491704287796932, 0.5900055454897571, 0.5365443722516139, 0.4974745943017082, 0.4539738602532766, 0.42801936625858433, 0.40088720040378334, 0.3771319383584359, 0.35403382453092813, 0.35102530188636866, 0.32766120269527094, 0.31244271107826893, 0.3111038965379316, 0.30783928055936616, 0.29716538617606686, 0.292320988145973, 0.28721707698621735, 0.2796935978628072, 0.2698164525800099, 0.26621280768552325, 0.2597349639494183, 0.2531122679386677, 0.24464098065371676, 0.2567716796692776, 0.2622098675930194, 0.249026354854112